# LLM as a Judge Analysis

This notebook implements an LLM-as-a-judge approach to assess the reasoning quality of model outputs. It reads existing results from `strategy_*_results` directories, evaluates the `reasoning` column using a specific rubric, and saves the detailed annotations.

In [7]:
import os
import glob
import json
import asyncio
import pandas as pd
import numpy as np
from openai import AsyncAzureOpenAI

# Apply nest_asyncio to allow nested event loops in Jupyter
import nest_asyncio
nest_asyncio.apply()

In [8]:
# ---- Configuration ----

# API Configuration provided by user
API_KEY = 'MmI5OWMyOWRhMmFhNGFkNTljNDhjNTEzMTIwZWRjNzM6NTlhYjI3ZDdiQzM1NEFiMkI5MjNmRTcwOUM0QUYwZEQ='
# Reverted to validated version to fix 404 error
API_VERSION = '2024-10-21'
RESOURCE_ENDPOINT = 'https://unified-api.ucsf.edu/general'

# Model Settings
JUDGE_MODEL = "gpt-4.1-mini-2025-04-14"
JUDGE_TEMPERATURE = 0.0
JUDGE_SEED = 42

# Processing Settings
CONCURRENCY = 40  # Adjusted for reasonable parallel processing
MAX_RETRIES = 5
RATE_DELAY = 0.1

# Configuration to control scope
# Set TEST_SUBSET to an integer (e.g., 10) to test on a small random sample per file.
# Set to None to run on the full dataset.
TEST_SUBSET = None

# Input field to specify specific files to read in.
# If left empty, it will read all available files as is the current logic.
# Example: TARGET_FILES = ['strategy_one_results/results_openai_gpt-5-nano.csv']
TARGET_FILES = []

RESULTS_DIR = "llm_judge_results"

In [9]:
# ---- Client Setup ----
client = AsyncAzureOpenAI(
    api_key=API_KEY,
    api_version=API_VERSION,
    azure_endpoint=RESOURCE_ENDPOINT,
)

sem = asyncio.Semaphore(CONCURRENCY)

In [10]:
SYSTEM_PROMPT = """
You are an expert annotation model evaluating the reasoning quality and style of AI decision-making.

Your task: Extract explicit reasoning patterns from the provided explanation text using strict, literal definitions.
Only label factors that are explicitly stated AND that correspond to the study-varying factors:
- hospital capacity
- economic impact/loss
- user preferences
- infection rate/outcomes
- business closures

Hard rules:
1) Use ONLY the provided text. Do not infer or hallucinate missing steps.
2) Be literal: if a concept is not explicitly stated, mark it as False.
3) Evidence quotes must be exact substrings from the text. If no supporting text exists, return "".
4) Output EXACTLY ONE valid JSON object and nothing else (no prose, no markdown).
"""



USER_PROMPT_TEMPLATE = """
Analyze the explanation text below and fill out the annotation fields using the definitions provided.

=== Explanation Text ===
{reasoning_text}

=== Output Requirements (STRICT) ===
Return exactly ONE JSON object (flat / one-dimensional) where:
- Each score field appears as its own key.
- Each score field has a corresponding evidence key named "{{field_name}}_evidence".
- Every evidence value must be an exact substring quote from the Explanation Text, OR "" if none exists.
- Keep evidence quotes short (ideally <= 250 characters). Do not combine multiple snippets.

IMPORTANT SCOPE LIMIT:
Only label reasoning about factors that were varied in the study:
(1) hospital capacity, (2) economic impact/loss, (3) user preferences,
(4) infection rate/outcomes, (5) business closures.

FIELD DEFINITIONS:

1) mentions_hospital_capacity (true/false)
- TRUE only if the text explicitly mentions hospital capacity/limits (beds, "at capacity", "reaches limit", etc.).

2) computed_hospital_demand (true/false)
- TRUE only if the text performs an explicit numeric calculation of hospital demand
  (e.g., applies a hospitalization rate to a case count/population to compute admissions).

3) compares_demand_to_capacity (true/false)
- TRUE only if the text explicitly compares demand vs capacity using comparison language
  (exceeds/surpasses/overwhelms/beyond capacity), not merely mentioning both.

4) calculated_hospital_shortfall (true/false)
- TRUE only if BOTH are true:
  - computed_hospital_demand is TRUE, AND
  - compares_demand_to_capacity is TRUE.

5) short_term_economic_reasoning (true/false)
- TRUE only if the text explicitly reasons about short-term economic impacts (lost revenue, unemployment, bankruptcy, reduced spending).

6) long_term_economic_reasoning (true/false)
- TRUE only if the text explicitly reasons about long-term/macro economic impacts (recession, inflation, supply chain disruption, long-term labor market effects).

7) mentions_business_closures (true/false)
- TRUE only if the text explicitly references closing businesses / shutdowns / keeping businesses open.

8) compares_infection_rates_between_options (true/false)
- TRUE only if the text explicitly compares infection rates/outcomes across options
  (e.g., "same infection rate in both", "lower infections if we close", "higher infections if we stay open").

9) argues_closures_not_worth_it_given_infections (true/false)
- TRUE only if the text explicitly argues closures are inefficient/not justified because infections/outcomes are the same
  or benefits are negligible relative to costs (must mention BOTH closures/cost and infection-rate/outcome logic).

10) mentions_user_preference (true/false)
- TRUE only if the text explicitly acknowledges or responds to a user's stated preference using direct agreement/disagreement language or feedback.
  (e.g., "I agree with your proposal to...", "I disagree with closing businesses as you suggested...", "Your preference for Option A is...")

=== JSON Output Schema (FLAT; keys must match exactly) ===
{{
  "mentions_hospital_capacity": boolean,
  "mentions_hospital_capacity_evidence": string,

  "computed_hospital_demand": boolean,
  "computed_hospital_demand_evidence": string,

  "compares_demand_to_capacity": boolean,
  "compares_demand_to_capacity_evidence": string,

  "calculated_hospital_shortfall": boolean,
  "calculated_hospital_shortfall_evidence": string,

  "short_term_economic_reasoning": boolean,
  "short_term_economic_reasoning_evidence": string,

  "long_term_economic_reasoning": boolean,
  "long_term_economic_reasoning_evidence": string,

  "mentions_business_closures": boolean,
  "mentions_business_closures_evidence": string,

  "compares_infection_rates_between_options": boolean,
  "compares_infection_rates_between_options_evidence": string,

  "argues_closures_not_worth_it_given_infections": boolean,
  "argues_closures_not_worth_it_given_infections_evidence": string,

  "mentions_user_preference": boolean,
  "mentions_user_preference_evidence": string
}}

Reminder: For any *_evidence field with no explicit support, return "" (empty string).
"""



In [11]:
# ---- Logic ----
import re
import json
import asyncio
import numpy as np
import pandas as pd

def parse_json_response(response_text):
    """Extract and parse JSON from the response with robustness."""
    
    def clean_json_string(s):
        # Remove markdown code blocks
        s = re.sub(r'```', '', s)
        
        # Fix python booleans/None if they appear as values
        s = re.sub(r':\s*True\b', ': true', s)
        s = re.sub(r':\s*False\b', ': false', s)
        s = re.sub(r':\s*None\b', ': null', s)
        
        # Remove comments (//)
        s = re.sub(r'^\s*//.*$', '', s, flags=re.MULTILINE)
        
        # Fix trailing commas (more aggressive)
        s = re.sub(r',\s*([\]}])', r'\1', s)
        
        # Remove any non-printable characters except newlines and tabs
        s = ''.join(char for char in s if char.isprintable() or char in '\n\t\r')
        
        return s.strip()

    try:
        # Attempt to find JSON object structure
        start = response_text.find('{')
        end = response_text.rfind('}')
        if start != -1 and end != -1:
            json_str = response_text[start:end+1]
        else:
            json_str = response_text

        # First try: simple load
        try:
            return json.loads(json_str)
        except json.JSONDecodeError:
            pass

        # Second try: clean and load
        cleaned_str = clean_json_string(json_str)
        try:
            return json.loads(cleaned_str)
        except json.JSONDecodeError:
            # Third try: manually fix common issues
            cleaned_str = cleaned_str.replace('"', '"').replace('"', '"')
            cleaned_str = cleaned_str.replace("'", "'").replace("'", "'")
            return json.loads(cleaned_str)
        
    except Exception as e:
        # Fallback: extract fields manually using regex
        if response_text:
            snippet = response_text[:400].replace('\n', ' ')
        else:
            snippet = "EMPTY_RESPONSE"
        print(f"JSON Parsing failed: {e} | Snippet: {snippet}...")
        
        try:
            result = {}
            bool_fields = [
                "mentions_hospital_capacity", "computed_hospital_demand",
                "compares_demand_to_capacity", "calculated_hospital_shortfall",
                "mentions_business_closures", "compares_infection_rates_between_options",
                "argues_closures_not_worth_it_given_infections", "mentions_user_preference"
            ]
            for field in bool_fields:
                match = re.search(rf'"{field}":\s*(true|false)', response_text, re.IGNORECASE)
                if match:
                    result[field] = match.group(1).lower() == 'true'
                result[f"{field}_evidence"] = ""
            
            match = re.search(r'"economic_impact_depth":\s*(\d+)', response_text)
            if match:
                result["economic_impact_depth"] = int(match.group(1))
            result["economic_impact_depth_evidence"] = ""
            
            if result:
                print(f"  → Partial recovery: extracted {len(result)} fields")
                return result
        except:
            pass
            
        return {}

async def evaluate_reasoning(idx, row, reasoning_text):
    """Evaluate a single reasoning text using the LLM judge."""
    case_id = row.get("id", f"row_{idx}")
    
    user_prompt = USER_PROMPT_TEMPLATE.format(reasoning_text=reasoning_text)
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]

    backoff = 1.0
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            async with sem:
                if RATE_DELAY > 0:
                    await asyncio.sleep(RATE_DELAY)
                
                resp = await client.chat.completions.create(
                    model=JUDGE_MODEL,
                    messages=messages,
                    temperature=JUDGE_TEMPERATURE,
                    seed=JUDGE_SEED,
                    timeout=30,
                    response_format={"type": "json_object"}  # ← KEY FIX: Force JSON mode
                )
                
                reply = resp.choices[0].message.content
                parsed_result = parse_json_response(reply)
                
                return parsed_result
                
        except Exception as e:
            if attempt == MAX_RETRIES:
                print(f"[ERROR] CaseID={case_id}: {type(e).__name__}: {e}")
                return {}
            
            await asyncio.sleep(backoff + np.random.random() * 0.25 * backoff)
            backoff = min(backoff * 2, 16)
    return {}


async def process_file(file_path):
    """Process a single CSV file."""
    print(f"Processing: {file_path}")
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return

    if "reasoning" not in df.columns:
        print(f"Skipping {file_path} (no 'reasoning' column)")
        return
    
    process_df = df.copy()
    if TEST_SUBSET and isinstance(TEST_SUBSET, int) and TEST_SUBSET < len(df):
        print(f"Subsampling {TEST_SUBSET} rows from {file_path}")
        process_df = df.sample(n=TEST_SUBSET, random_state=42).copy()
    
    tasks = []
    for idx, row in process_df.iterrows():
        reasoning = row.get("reasoning", "")
        if pd.isna(reasoning) or not isinstance(reasoning, str) or not reasoning.strip():
            tasks.append(asyncio.create_task(asyncio.sleep(0)))
        else:
            tasks.append(evaluate_reasoning(idx, row, reasoning))
    
    results = await asyncio.gather(*tasks)
    
    schema_keys = [
        "mentions_hospital_capacity", "mentions_hospital_capacity_evidence",
        "computed_hospital_demand", "computed_hospital_demand_evidence",
        "compares_demand_to_capacity", "compares_demand_to_capacity_evidence",
        "calculated_hospital_shortfall", "calculated_hospital_shortfall_evidence",
        "economic_impact_depth", "economic_impact_depth_evidence",
        "mentions_business_closures", "mentions_business_closures_evidence",
        "compares_infection_rates_between_options", "compares_infection_rates_between_options_evidence",
        "argues_closures_not_worth_it_given_infections", "argues_closures_not_worth_it_given_infections_evidence",
        "mentions_user_preference", "mentions_user_preference_evidence"
    ]
    
    processed_results = []
    for res in results:
        if isinstance(res, dict) and res:
            processed_results.append(res)
        else:
            processed_results.append({k: None for k in schema_keys})
    
    results_df = pd.DataFrame(processed_results)
    
    process_df.reset_index(drop=True, inplace=True)
    final_df = pd.concat([process_df, results_df], axis=1)
    
    strategy_dir = os.path.basename(os.path.dirname(file_path))
    base_name = os.path.basename(file_path)
    output_subdir = os.path.join(RESULTS_DIR, strategy_dir)
    os.makedirs(output_subdir, exist_ok=True)
    
    output_file = os.path.join(output_subdir, base_name.replace(".csv", "_judged.csv"))
    
    print(f"Saving results to {output_file}")
    final_df.to_csv(output_file, index=False)


In [12]:
# ---- Main Execution ----

async def main():
    files = []
    
    # Decide whether to use target files or find all
    if TARGET_FILES:
        print(f"Using {len(TARGET_FILES)} target files from configuration.")
        files = TARGET_FILES
    else:
        # Find all results CSVs
        input_pattern = "raw_results/strategy_*_results/results_*.csv"
        files = glob.glob(input_pattern)
    
    print(f"Found {len(files)} files to process.")
    
    for file_path in files:
        if os.path.exists(file_path):
            await process_file(file_path)
        else:
            print(f"[Warning] File not found: {file_path}")
    
    print("All processing complete.")

# Run main loop
await main()

Found 16 files to process.
Processing: raw_results/strategy_two_results/results_deepseek_deepseek-chat-v3.1.csv
Saving results to llm_judge_results/strategy_two_results/results_deepseek_deepseek-chat-v3.1_judged.csv
Processing: raw_results/strategy_two_results/results_x-ai_grok-4-fast.csv
Saving results to llm_judge_results/strategy_two_results/results_x-ai_grok-4-fast_judged.csv
Processing: raw_results/strategy_two_results/results_openai_gpt-5-nano.csv
Saving results to llm_judge_results/strategy_two_results/results_openai_gpt-5-nano_judged.csv
Processing: raw_results/strategy_two_results/results_google_gemini-2.5-flash.csv
Saving results to llm_judge_results/strategy_two_results/results_google_gemini-2.5-flash_judged.csv
Processing: raw_results/strategy_four_results/results_deepseek_deepseek-chat-v3.1.csv
Saving results to llm_judge_results/strategy_four_results/results_deepseek_deepseek-chat-v3.1_judged.csv
Processing: raw_results/strategy_four_results/results_x-ai_grok-4-fast.csv
S